# Test MoCaXExtend sous Windows (Python 3.8, Miniforge)
Noyau : **Python (mocax38)**. Ce notebook doit être dans le même dossier que `mocax_extend_loader.py` et `compat.py`.

In [1]:
import os, sys
_env_bins = [os.path.join(sys.prefix, *p) for p in
             (("Library", "mingw-w64", "bin"), ("Library", "usr", "bin"),
              ("Library", "bin"), ("Scripts",), ("bin",))]
os.environ["PATH"] = os.pathsep.join([p for p in _env_bins if os.path.isdir(p)] + [os.environ["PATH"]])
os.add_dll_directory(os.path.join(sys.prefix, "Library", "bin"))

import os, sys, struct, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, scipy
print(sys.version)
print("64 bits :", struct.calcsize("P") == 8, "| numpy", np.__version__, "| scipy", scipy.__version__)

MOCAX_ROOT = r"C:\Users\ezzahid\Downloads\MoCaXExtend"   # dossier qui contient Windows\Library
EXAMPLES = os.path.join(MOCAX_ROOT, "Windows", "Examples")


3.8.20 | packaged by conda-forge | (default, Sep 30 2024, 17:44:03) [MSC v.1929 64 bit (AMD64)]
64 bits : True | numpy 1.24.4 | scipy 1.10.1


In [2]:
from mocax_extend_loader import load_mocax_extend
me = load_mocax_extend(search_roots=[MOCAX_ROOT], verbose=True)
import scikit_tt
print("scikit_tt OK")

mocaxextendpy importe depuis C:\Users\ezzahid\Downloads\MoCaXExtend\Windows\Library
  natif tensorvals : C:\Users\ezzahid\Downloads\MoCaXExtend\Windows\Library\shared_libs\tensorvals.dll
  natif hommat     : C:\Users\ezzahid\Downloads\MoCaXExtend\Windows\Library\shared_libs\hommat.dll
  natif gslcblas   : C:\Users\ezzahid\Downloads\MoCaXExtend\Windows\Library\shared_libs\gslcblas.dll
scikit_tt OK


## 1. test_release.py (éditeur) : exp(-|x|), d=4

In [3]:
def funct(x): return np.exp(-np.linalg.norm(x, axis=1))

dimension = 4
num_cheb_pts = dimension * [13]
variable_ranges = dimension * [[0, 1]]

obj = me.MocaxExtend(dimension, num_cheb_pts, variable_ranges)
random_cheb_pts = obj.subgrid_by_number(1000)
obj.set_subgrid_values(funct(random_cheb_pts))
obj.gen_train_val_data()
print("Sous-grille : {:.5f} % de la grille".format(100 * obj.get_subgrid_size() / obj.get_tensor_size()))

t0 = time.perf_counter()
obj.run_rank_adaptive_algo(tolerance=1e-3, rel_tolerance=1e-8, max_iters=100,
                           max_rank=10, print_progress=True, max_rounds=5)
print("Fit : %.1f s" % (time.perf_counter() - t0))

Sous-grille : 3.47327 % de la grille


	At iteration: 10
	Relative error change training: 9.306e-16
	Relative error change testing: 3.820e-09
	Leaving algo

_________________________________________

Current tensor: 
Tensor train with order    = 4, 
                  row_dims = [14, 14, 14, 14], 
                  col_dims = [1, 1, 1, 1], 
                  ranks    = [1, 1, 1, 1, 1]

Trying increase rank 1 from 1 to 2


	At iteration: 14
	Relative error change training: 1.114e-12
	Relative error change testing: 9.300e-09
	Leaving algo


Current train cost: 6.849e-02
Current validation cost: 8.879e-02

Error improvement of -5.390e-03 in rank increase attempt

SUCCEEDED with completion algo. Updating to new tensor

Current max rank 2
_________________________________________

Current tensor: 
Tensor train with order    = 4, 
                  row_dims = [14, 14, 14, 14], 
                  col_dims = [1, 1, 1, 1], 
                  ranks    = [1, 2, 1, 1, 1]

Trying increase rank 2 fro

In [5]:
fn = os.path.join(os.environ.get("TEMP", "."), "tensor_extension_obj.pickle")
obj.serialize(fn)
obj_des = me.MocaxExtend.deserialize(fn)

Q = np.random.default_rng(1).uniform(0, 1, size=(1000, dimension))
t0 = time.perf_counter(); v = obj.cheb_tensor_evals(Q); t = time.perf_counter() - t0
err = np.abs(v - funct(Q))
print("Évaluation de 1000 points : %.5f s" % t)
print("Erreur max %.2e | moyenne %.2e" % (err.max(), err.mean()))
print("Écart objet relu : %.1e" % np.abs(np.asarray(obj_des.cheb_tensor_evals(Q)) - v).max())
print("TEST 1 OK")

Évaluation de 1000 points : 0.64726 s
Erreur max 6.60e-02 | moyenne 1.97e-03
Écart objet relu : 2.8e-16
TEST 1 OK


## 2. Black-Scholes, d=5 (cas commenté de test_release.py, via black_scholes_utils.py)

In [7]:
sys.path.insert(0, EXAMPLES)
import black_scholes_utils as bs

def funct_bs(x): return np.array([bs.blackScholes("C", *pt) for pt in x])

R = [[50, 150], [50, 150], [0.01, 0.1], [0.2, 0.7], [1.0, 3.0]]
o = me.MocaxExtend(5, 5 * [10], R)
p = np.asarray(o.subgrid_by_number(2000), dtype=float)
o.set_subgrid_values(funct_bs(p))
o.gen_train_val_data()
o.run_rank_adaptive_algo(tolerance=1e-3, rel_tolerance=1e-8, max_iters=100,
                         max_rank=10, print_progress=False, max_rounds=5)
lo, hi = np.array(R, dtype=float).T
Q = np.random.default_rng(2).uniform(lo, hi, size=(1000, 5))
e = np.abs(o.cheb_tensor_evals(Q) - funct_bs(Q))
print("Erreur max %.3f | moyenne %.4f (prix entre 0 et ~110)" % (e.max(), e.mean()))
print("TEST 2 OK")

Erreur max 0.632 | moyenne 0.0703 (prix entre 0 et ~110)
TEST 2 OK
